In [ ]:
# ============================================
# 实验1: 数据加载与空间图构建
# ============================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.neighbors import kneighbors_graph
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

In [ ]:
# ============================================
# 1. 加载导出的数据
# ============================================
print("\n" + "=" * 60)
print("1. 加载数据")
print("=" * 60)

# 加载GNN专用数据
df = pd.read_csv("./data/xenium_for_gnn.csv")
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"\n数据预览:")
print(df.head())

# 检查缺失值
print(f"\n缺失值统计:")
print(df.isnull().sum())

# 删除有缺失值的行
df = df.dropna()
print(f"删除缺失值后数据形状: {df.shape}")

In [ ]:
# ============================================
# 2. 提取特征和标签
# ============================================
print("\n" + "=" * 60)
print("2. 提取特征和标签")
print("=" * 60)

# 提取空间坐标
coords = df[["x_centroid", "y_centroid"]].values
print(f"坐标矩阵形状: {coords.shape}")

# 提取PCA特征
pca_cols = [col for col in df.columns if col.startswith("PC_")]
features = df[pca_cols].values
print(f"PCA特征形状: {features.shape}")

# 标准化特征
scaler = StandardScaler()
features = scaler.fit_transform(features)

# 提取标签
labels_raw = df["predicted_cell_type"].values
print(f"\n原始标签示例: {np.unique(labels_raw)[:10]}")
print(f"总类别数: {len(np.unique(labels_raw))}")

# 标签编码
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_raw)
num_classes = len(label_encoder.classes_)
print(f"编码后类别数: {num_classes}")
print(f"类别映射:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

# 统计各类别样本数
print(f"\n各类别样本数:")
class_counts = pd.Series(labels).value_counts().sort_index()
for i, count in class_counts.items():
    print(f"  类别 {i} ({label_encoder.classes_[i]}): {count}")

In [ ]:
# ============================================
# 3. 构建空间邻接图
# ============================================
print("\n" + "=" * 60)
print("3. 构建空间邻接图")
print("=" * 60)


def build_spatial_graph(coords, k=10, mode="knn"):
    """
    构建空间邻接图

    参数:
        coords: 空间坐标矩阵 (n_cells, 2)
        k: 邻居数量
        mode: 'knn' 或 'radius'
    """
    if mode == "knn":
        # K近邻图
        adj_matrix = kneighbors_graph(
            coords, n_neighbors=k, mode="connectivity", include_self=False
        )
    elif mode == "radius":
        # 半径图（自动计算合适的半径）
        from sklearn.neighbors import radius_neighbors_graph

        # 计算平均最小距离
        from sklearn.neighbors import NearestNeighbors

        nbrs = NearestNeighbors(n_neighbors=2).fit(coords)
        distances, _ = nbrs.kneighbors(coords)
        mean_min_dist = distances[:, 1].mean()
        radius = mean_min_dist * 2
        print(f"  计算得到的半径: {radius:.2f}")
        adj_matrix = radius_neighbors_graph(
            coords, radius=radius, mode="connectivity", include_self=False
        )
    else:
        raise ValueError("mode must be 'knn' or 'radius'")

    # 转换为PyTorch Geometric格式
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_matrix)

    return edge_index


# 尝试不同的k值
k_values = [5, 10, 15, 20, 30]

for k in k_values:
    print(f"\n尝试 k={k}:")
    edge_index = build_spatial_graph(coords, k=k, mode="knn")
    print(f"  边数量: {edge_index.shape[1]}")
    print(f"  平均度: {2 * edge_index.shape[1] / coords.shape[0]:.2f}")

# 选择默认k值
DEFAULT_K = 15
edge_index = build_spatial_graph(coords, k=DEFAULT_K, mode="knn")
print(f"\n使用 k={DEFAULT_K} 构建图")
print(f"边数量: {edge_index.shape[1]}")
print(f"平均度: {2 * edge_index.shape[1] / coords.shape[0]:.2f}")

In [ ]:
# ============================================
# 4. 创建PyTorch Geometric数据对象
# ============================================
print("\n" + "=" * 60)
print("4. 创建PyTorch Geometric数据对象")
print("=" * 60)

# 转换为Tensor
x = torch.FloatTensor(features)
y = torch.LongTensor(labels)
edge_index = edge_index.to(torch.long)

# 创建数据对象
data = Data(x=x, edge_index=edge_index, y=y)

print(f"数据对象:")
print(f"  节点数: {data.num_nodes}")
print(f"  特征维度: {data.num_features}")
print(f"  边数: {data.num_edges}")
print(f"  类别数: {num_classes}")

In [ ]:
# ============================================
# 5. 划分训练集、验证集、测试集
# ============================================
print("\n" + "=" * 60)
print("5. 划分数据集")
print("=" * 60)

# 首先划分训练+验证集和测试集
indices = np.arange(data.num_nodes)
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)

# 再从训练+验证集中划分训练集和验证集
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, stratify=labels[train_val_idx], random_state=42
)

# 创建训练/验证/测试掩码
train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"训练集大小: {train_mask.sum().item()}")
print(f"验证集大小: {val_mask.sum().item()}")
print(f"测试集大小: {test_mask.sum().item()}")

In [ ]:
# ============================================
# 6. 定义图神经网络模型
# ============================================
print("\n" + "=" * 60)
print("6. 定义图神经网络模型")
print("=" * 60)


class GCN(nn.Module):
    """图卷积网络"""

    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)

        return F.log_softmax(x, dim=1)


class GraphSAGE(nn.Module):
    """GraphSAGE模型"""

    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)

        return F.log_softmax(x, dim=1)


class GAT(nn.Module):
    """图注意力网络"""

    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.5):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = GATConv(
            hidden_dim * heads, out_dim, heads=1, concat=False, dropout=dropout
        )
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)


class MLP(nn.Module):
    """多层感知机（基线模型，不使用图结构）"""

    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.5):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, data):
        x = data.x

        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.fc2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.fc3(x)

        return F.log_softmax(x, dim=1)

In [ ]:
# ============================================
# 7. 训练函数
# ============================================
print("\n" + "=" * 60)
print("7. 模型训练")
print("=" * 60)


def train_model(model, data, optimizer, epochs=200, patience=20, verbose=True):
    """
    训练模型

    参数:
        model: PyTorch模型
        data: PyTorch Geometric数据对象
        optimizer: 优化器
        epochs: 最大训练轮数
        patience: 早停轮数
        verbose: 是否打印训练信息
    """
    model = model.to(device)
    data = data.to(device)

    best_val_loss = float("inf")
    best_val_acc = 0
    best_epoch = 0
    patience_counter = 0

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        # 训练阶段
        model.train()
        optimizer.zero_grad()

        out = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

        # 计算训练准确率
        pred = out.argmax(dim=1)
        train_acc = (
            (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
        )

        # 验证阶段
        model.eval()
        with torch.no_grad():
            out = model(data)
            val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask])
            val_acc = (
                (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            )

        # 记录
        train_losses.append(loss.item())
        val_losses.append(val_loss.item())
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        # 早停检查
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss.item()
            best_epoch = epoch
            patience_counter = 0
            # 保存最佳模型
            best_model_state = model.state_dict()
        else:
            patience_counter += 1

        if patience_counter >= patience:
            if verbose:
                print(f"  早停于 epoch {epoch+1}")
            break

        # 打印训练信息
        if verbose and (epoch + 1) % 20 == 0:
            print(
                f"  Epoch {epoch+1:3d}/{epochs} | "
                f"Train Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss.item():.4f} | Val Acc: {val_acc:.4f}"
            )

    # 加载最佳模型
    model.load_state_dict(best_model_state)

    if verbose:
        print(f"\n最佳验证结果: Epoch {best_epoch+1}, Val Acc: {best_val_acc:.4f}")

    return model, train_losses, val_losses, train_accs, val_accs


def evaluate_model(model, data):
    """评估模型"""
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)

        # 测试集评估
        test_pred = pred[data.test_mask].cpu().numpy()
        test_true = data.y[data.test_mask].cpu().numpy()

        test_acc = accuracy_score(test_true, test_pred)
        test_f1 = f1_score(test_true, test_pred, average="macro")

        return test_acc, test_f1, test_pred, test_true

In [ ]:
# ============================================
# 8. 训练不同模型并进行对比
# ============================================
print("\n" + "=" * 60)
print("8. 模型对比实验")
print("=" * 60)

# 模型配置
configs = {
    "MLP": {"model_class": MLP, "params": {"hidden_dim": 128, "dropout": 0.5}},
    "GCN": {"model_class": GCN, "params": {"hidden_dim": 128, "dropout": 0.5}},
    "GraphSAGE": {
        "model_class": GraphSAGE,
        "params": {"hidden_dim": 128, "dropout": 0.5},
    },
    "GAT": {
        "model_class": GAT,
        "params": {"hidden_dim": 64, "heads": 4, "dropout": 0.5},
    },
}

in_dim = data.num_features
out_dim = num_classes

results = {}
models = {}

for model_name, config in configs.items():
    print(f"\n{'-'*50}")
    print(f"训练 {model_name} 模型")
    print(f"{'-'*50}")

    # 创建模型
    if model_name == "GAT":
        model = config["model_class"](
            in_dim,
            config["params"]["hidden_dim"],
            out_dim,
            heads=config["params"]["heads"],
            dropout=config["params"]["dropout"],
        )
    else:
        model = config["model_class"](
            in_dim,
            config["params"]["hidden_dim"],
            out_dim,
            dropout=config["params"]["dropout"],
        )

    # 优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # 训练
    trained_model, train_losses, val_losses, train_accs, val_accs = train_model(
        model, data, optimizer, epochs=200, patience=30, verbose=True
    )

    # 评估
    test_acc, test_f1, test_pred, test_true = evaluate_model(trained_model, data)

    results[model_name] = {
        "test_acc": test_acc,
        "test_f1": test_f1,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accs": train_accs,
        "val_accs": val_accs,
        "test_pred": test_pred,
        "test_true": test_true,
    }
    models[model_name] = trained_model

    print(f"\n{model_name} 测试结果:")
    print(f"  准确率: {test_acc:.4f}")
    print(f"  Macro F1: {test_f1:.4f}")

In [ ]:
# ============================================
# 9. 结果对比可视化
# ============================================
print("\n" + "=" * 60)
print("9. 结果可视化")
print("=" * 60)

# 设置绘图样式
plt.style.use("seaborn-v0_8-darkgrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 9.1 训练曲线对比
ax = axes[0, 0]
for model_name, result in results.items():
    ax.plot(
        result["train_accs"], label=f"{model_name} (Train)", linestyle="--", alpha=0.7
    )
    ax.plot(result["val_accs"], label=f"{model_name} (Val)", linestyle="-")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("训练曲线对比")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# 9.2 测试准确率对比
ax = axes[0, 1]
model_names = list(results.keys())
test_accs = [results[m]["test_acc"] for m in model_names]
colors = ["#2ecc71", "#3498db", "#9b59b6", "#e74c3c"]
bars = ax.bar(model_names, test_accs, color=colors[: len(model_names)])
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("测试集准确率对比")
# 添加数值标签
for bar, acc in zip(bars, test_accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{acc:.4f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )

# 9.3 损失曲线对比
ax = axes[1, 0]
for model_name, result in results.items():
    ax.plot(result["val_losses"], label=f"{model_name}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("验证损失曲线对比")
ax.legend()
ax.grid(True, alpha=0.3)

# 9.4 混淆矩阵（以GCN为例）
ax = axes[1, 1]
best_model_name = max(results.keys(), key=lambda x: results[x]["test_acc"])
best_result = results[best_model_name]

# 计算混淆矩阵
cm = confusion_matrix(best_result["test_true"], best_result["test_pred"])
# 归一化
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

# 只显示前10个类别（避免过于拥挤）
top_10_classes = np.argsort(np.bincount(best_result["test_true"]))[-10:]
cm_subset = cm_norm[top_10_classes][:, top_10_classes]

# 获取类别标签
class_labels = [label_encoder.classes_[i] for i in top_10_classes]
# 简化标签名称（取最后部分）
short_labels = [
    label.split()[-1][:15] if " " in label else label[:15] for label in class_labels
]

sns.heatmap(
    cm_subset,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=short_labels,
    yticklabels=short_labels,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"{best_model_name} 混淆矩阵 (Top 10 类别)")
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.setp(ax.yaxis.get_majorticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("./results/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 对比图已保存: ./results/model_comparison.png")

In [ ]:
# ============================================
# 10. 打印详细结果报告
# ============================================
print("\n" + "=" * 60)
print("10. 详细结果报告")
print("=" * 60)

print("\n📊 模型性能对比:")
print("-" * 50)
print(f"{'Model':<12} {'Test Acc':<12} {'Macro F1':<12}")
print("-" * 50)
for model_name, result in results.items():
    print(f"{model_name:<12} {result['test_acc']:<12.4f} {result['test_f1']:<12.4f}")
print("-" * 50)

# 找出最佳模型
best_model_name = max(results.keys(), key=lambda x: results[x]["test_acc"])
print(f"\n🏆 最佳模型: {best_model_name}")
print(f"   测试准确率: {results[best_model_name]['test_acc']:.4f}")
print(f"   Macro F1: {results[best_model_name]['test_f1']:.4f}")

# 打印GCN的分类报告（作为示例）
gcn_result = results["GCN"]
print(f"\n📋 GCN 分类报告:")
print(
    classification_report(
        gcn_result["test_true"],
        gcn_result["test_pred"],
        target_names=label_encoder.classes_,
        zero_division=0,
    )
)

In [ ]:
# ============================================
# 11. 空间可视化（使用最佳模型）
# ============================================
print("\n" + "=" * 60)
print("11. 空间可视化")
print("=" * 60)

# 获取最佳模型的预测结果
best_model = models[best_model_name]
best_model.eval()
data = data.to(device)

with torch.no_grad():
    out = best_model(data)
    all_pred = out.argmax(dim=1).cpu().numpy()

# 获取原始标签
true_labels = data.y.cpu().numpy()

# 创建空间可视化图
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 获取坐标
coords_2d = coords

# 11.1 真实标签空间分布
ax = axes[0]
scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1], c=true_labels, cmap="tab20", s=1, alpha=0.6
)
ax.set_title("真实细胞类型分布")
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_aspect("equal")
plt.colorbar(scatter, ax=ax, label="Cell Type", ticks=range(0, num_classes, 5))

# 11.2 预测标签空间分布
ax = axes[1]
scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1], c=all_pred, cmap="tab20", s=1, alpha=0.6
)
ax.set_title(f"{best_model_name} 预测细胞类型分布")
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_aspect("equal")
plt.colorbar(scatter, ax=ax, label="Cell Type", ticks=range(0, num_classes, 5))

# 11.3 错误分类点
ax = axes[2]
errors = all_pred != true_labels
error_colors = np.where(errors, 1, 0)
scatter = ax.scatter(
    coords_2d[:, 0], coords_2d[:, 1], c=error_colors, cmap="Reds", s=1, alpha=0.6
)
ax.set_title(f"错误分类点 (错误率: {errors.mean():.2%})")
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_aspect("equal")
plt.colorbar(scatter, ax=ax, label="Error", ticks=[0, 1])

plt.tight_layout()
plt.savefig("./results/spatial_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 空间可视化图已保存: ./results/spatial_visualization.png")

In [ ]:
# ============================================
# 12. 保存结果
# ============================================
print("\n" + "=" * 60)
print("12. 保存结果")
print("=" * 60)

# 保存预测结果
predictions_df = pd.DataFrame(
    {
        "cell_id": df["cell_id"],
        "x_centroid": coords[:, 0],
        "y_centroid": coords[:, 1],
        "true_label": label_encoder.inverse_transform(true_labels),
        "predicted_label": label_encoder.inverse_transform(all_pred),
        "is_correct": (all_pred == true_labels),
    }
)

# 添加各模型的预测结果
for model_name, result in results.items():
    predictions_df[f"{model_name}_pred"] = label_encoder.inverse_transform(
        result["test_pred"]
    )

predictions_df.to_csv("./results/final_predictions.csv", index=False)
print("✅ 预测结果已保存: ./results/final_predictions.csv")

# 保存结果汇总
summary_df = pd.DataFrame(
    [
        {
            "Model": model_name,
            "Test_Accuracy": result["test_acc"],
            "Macro_F1": result["test_f1"],
            "Best_Epoch": len(result["train_losses"]),
        }
        for model_name, result in results.items()
    ]
)
summary_df.to_csv("./results/model_summary.csv", index=False)
print("✅ 结果汇总已保存: ./results/model_summary.csv")

# 保存配置信息
config_df = pd.DataFrame(
    {
        "Parameter": [
            "k_neighbors",
            "hidden_dim",
            "dropout",
            "num_classes",
            "num_nodes",
        ],
        "Value": [DEFAULT_K, 128, 0.5, num_classes, data.num_nodes],
    }
)
config_df.to_csv("./results/experiment_config.csv", index=False)
print("✅ 配置信息已保存: ./results/experiment_config.csv")

print("\n" + "=" * 60)
print("实验完成！")
print("=" * 60)